# singular-matrix-mask-trick — worked example 2: Mask trick for 3×3 batch with one singular slice

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `singular-matrix-mask-trick`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The singular matrix mask trick generalizes to any square size n. The only change is passing `t.eye(n)` where n matches the matrix dimension. For 3×3 systems, a matrix is singular when the three row vectors lie in a plane (determinant = 0). The mask trick lets you handle mixed batches without splitting or looping.

## Worked solution

**Step 1 — Create a batch of 3×3 systems.** We build two well-conditioned systems and one singular one (last row = sum of first two rows).

**Step 2 — Detect singularity via det.** For exact singular matrices the determinant is exactly 0. For near-singular ones, we use the eps threshold.

**Step 3 — Clone and replace singular slices with I_3.** `t.eye(3)` is the 3×3 identity.

**Step 4 — Solve.** The patched slices will solve to `x = I^{-1} b = b`, which is meaningless — but the call won't crash.

**Step 5 — Report valid solutions.** We print the solutions for the two valid slices only.

In [ ]:
import torch as t

t.manual_seed(11)

def solve_with_mask_3x3(A, b, eps=1e-8):
    K, n, _ = A.shape
    dets = t.linalg.det(A)
    is_singular = dets.abs() < eps
    A_safe = A.clone()
    A_safe[is_singular] = t.eye(n, dtype=A.dtype)
    x = t.linalg.solve(A_safe, b)
    return x, ~is_singular

# 3 systems, one singular
A = t.zeros(3, 3, 3)
A[0] = t.tensor([[1.0,0.0,0.0],[0.0,2.0,0.0],[0.0,0.0,3.0]])  # diagonal
A[1] = t.tensor([[1.0,2.0,3.0],[0.0,1.0,4.0],[0.0,0.0,1.0]])  # upper triangular
A[2] = t.tensor([[1.0,2.0,3.0],[4.0,5.0,6.0],[5.0,7.0,9.0]])  # singular: row2=row0+row1
b = t.tensor([[1.0,2.0,3.0],[0.0,1.0,0.0],[4.0,5.0,6.0]])

x, valid = solve_with_mask_3x3(A, b)
print('valid:', valid.tolist())   # [T, T, F]
print('det of singular slice:', t.linalg.det(A[2]).item())
for i in range(3):
    if valid[i]:
        residual = (A[i] @ x[i] - b[i]).abs().max().item()
        print(f'Slice {i}: residual={residual:.2e}')